# Colab runner: full benchmark (Runtime > Change runtime type > GPU)

Chạy theo thứ tự. Dự đoán, checkpoint và kết quả được lưu trên Google Drive (không commit lên Git).
Cấu hình và giao thức nằm trong `configs/`; notebook chỉ gọi các script của repo.

In [ ]:
# 1. GPU và Google Drive
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/VISUAL_ANOMALY_INSPECTION_OUTPUTS'
!mkdir -p {OUT}

In [ ]:
# 2. Lấy mã nguồn và cài thư viện. Đổi BRANCH thành main sau khi nhánh tích hợp được merge.
BRANCH = 'integration'
%cd /content
!rm -rf VISUAL-ANOMALY-INSPECTION
!git clone -b {BRANCH} https://github.com/KNV-DSEB/VISUAL-ANOMALY-INSPECTION.git
%cd /content/VISUAL-ANOMALY-INSPECTION
!pip install -q -r requirements.txt

In [ ]:
# 3. Dữ liệu (5 category), EDA và test
!python scripts/download_mvtec.py
!python scripts/eda_mvtec.py
!python -m pytest -q

In [ ]:
# 4. Huấn luyện và dự đoán cho cả 3 phương pháp (E-02). Ngưỡng lấy từ val, test chỉ được chấm điểm.
!python scripts/train_and_predict.py --method all --pred-dir {OUT}/predictions --checkpoint-dir {OUT}/checkpoints

In [ ]:
# 5. Đánh giá chung: bảng theo category và tổng thể (E-02, L-01)
!python scripts/evaluate.py --pred-dir {OUT}/predictions --out-dir {OUT}/results/metrics

In [ ]:
# 6. Phân tích lỗi (E-03) và gallery định vị (L-02)
!python scripts/analyze_errors.py --pred-dir {OUT}/predictions --out-dir {OUT}/results/metrics
!python scripts/generate_galleries.py --pred-dir {OUT}/predictions --out-dir {OUT}/results/figures

In [ ]:
# 7. Cải tiến đã khai báo trước cho PatchCore (I-02). Báo cáo cạnh baseline; không chọn bản thắng bằng kết quả test.
!python scripts/train_and_predict.py --method patchcore --config configs/experiments/patchcore_refined.yaml     --pred-dir {OUT}/predictions_refined --checkpoint-dir {OUT}/checkpoints_refined
!python scripts/evaluate.py --methods patchcore --pred-dir {OUT}/predictions_refined --out-dir {OUT}/results/metrics_refined

In [ ]:
# 8. Độ nhạy theo số ảnh normal và bảng đánh đổi báo động giả (I-01). Mất nhiều thời gian nhất.
!python scripts/run_sensitivity.py --pred-dir {OUT}/predictions     --sens-pred-dir {OUT}/predictions_sensitivity --results-dir {OUT}/results

In [ ]:
# 9. Đóng gói bảng và hình nhẹ để tải về
!cp results/eda_* {OUT}/results/ 2>/dev/null || true
!cd {OUT} && zip -r results_package.zip results/
print('Xong. Tải results_package.zip; predictions/ và checkpoints/ nằm trên Drive.')